# LIWC 2015 analysis

**Author:** [Johnnatan Messias](https://johnnatan-messias.github.io)  
**Date:** June 2026

## Background: what LIWC does

**Linguistic Inquiry and Word Count (LIWC)** is a dictionary-based text analysis method. It estimates how often a text uses words associated with linguistic, psychological, social, and topical categories. Examples include pronouns, positive and negative emotion, cognitive processes, social relationships, time orientation, work, money, health, and risk.

LIWC does not infer meaning like a contextual language model. It performs lexical matching:

1. The text is divided into tokens.
2. Each token is normalized and looked up in the LIWC dictionary.
3. Exact entries match one token, while wildcard entries match a prefix. For example, `abandon*` can match `abandon`, `abandoned`, and `abandonment`.
4. Every matching category receives one count for that token.
5. Counts are divided by the total number of word tokens to make texts of different lengths more comparable.

The main score calculated by this notebook is:

`category_percentage = 100 * category matches / total word tokens`

For example, a Positive Emotions score of `4.5%` means that 4.5 out of every 100 word tokens matched that category. It does **not** mean that the author was 4.5% positive or that 4.5% of sentences expressed positive emotion.

## How categories work

LIWC categories overlap. A word may belong to a broad parent category and one or more child categories. A positive-emotion word may increment both `affect (Affect)` and `posemo (Positive Emotions)`. Therefore:

- category percentages do not add up to 100%;
- category counts must not be summed to calculate the number of words;
- parent and child categories are related, not independent measurements;
- a high score means more category-related vocabulary, not necessarily a stronger psychological state.

## Reading the output

- **Words**: all word tokens used as the main denominator.
- **Dictionary-matched words**: tokens assigned to at least one category.
- **Dictionary coverage**: the percentage of word tokens matching any category. Coverage is a quality diagnostic, not a measure of psychological validity.
- **Count**: token matches for a category.
- **% words**: the main LIWC-style percentage to report.
- **% matched**: the percentage among matched words only; useful mainly for diagnostics.

## Interpretation guidelines

Interpret scores comparatively and in context. Percentages are most useful when comparing groups, time periods, authors, or conditions. Differences may reflect topic, genre, audience, demographics, platform conventions, or preprocessing choices as well as psychological processes.

For a corpus, preserve each document as an observation when possible. Concatenating all texts creates one corpus-level score and allows longer documents or prolific authors to dominate. Very short texts and categories with only a few matches are especially unstable.

Before drawing conclusions, inspect dictionary coverage, frequent unmatched words, and examples in context. Dictionary matching cannot reliably resolve negation, sarcasm, ambiguity, quotations, or every contextual meaning of a word.

**Do not interpret LIWC scores as diagnoses or direct measurements of personality, mental health, deception, intention, or emotional state.** They are lexical indicators that require a research question, an appropriate comparison, and supporting evidence.

## Scope and limitations

This notebook reproduces dictionary-category matching from the LIWC 2015 English `.dic` file. Its tokenizer may differ from the official application. Proprietary summary variables such as Analytic, Clout, Authentic, and Tone require LIWC's official algorithms and cannot be reconstructed exactly from the dictionary alone.


In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
from pathlib import Path
import re

DICTIONARY_PATH = Path("dictionary/LIWC2015_English.dic")
assert DICTIONARY_PATH.exists(
), f"Dictionary not found: {DICTIONARY_PATH.resolve()}"

## Dictionary parser and tokenizer

An entry ending in `*` is a prefix wildcard: for example, `abandon*` matches `abandon`, `abandoned`, and `abandonment`. Exact entries and all matching wildcard prefixes are combined. The tokenizer preserves contractions, hyphenated forms, and symbolic dictionary entries such as `:)`.


In [3]:
def load_liwc_dictionary(path):
    """Return category names, exact entries, and wildcard-prefix entries."""
    category_names = {}
    exact = {}
    prefixes = {}
    section = 0

    with Path(path).open(encoding="utf-8-sig") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                continue
            if line == "%":
                section += 1
                continue

            fields = line.split("\t")
            if section == 1:
                fields = [field.strip() for field in fields if field.strip()]
                if len(fields) >= 2 and fields[0].isdigit():
                    category_names[int(fields[0])] = fields[1]
            elif section == 2:
                token = fields[0].strip().lower().replace("’", "'")
                try:
                    category_ids = tuple(int(value)
                                         for value in fields[1:] if value.strip())
                except ValueError as error:
                    raise ValueError(
                        f"Invalid category on dictionary line {line_number}") from error
                unknown = set(category_ids) - set(category_names)
                if unknown:
                    raise ValueError(
                        f"Unknown categories {unknown} on dictionary line {line_number}")
                target = prefixes if token.endswith("*") else exact
                key = token[:-1] if token.endswith("*") else token
                target[key] = category_ids

    if section < 2 or not category_names or not exact:
        raise ValueError("The file does not look like a LIWC .dic file")
    return category_names, exact, prefixes


CATEGORY_NAMES, EXACT_ENTRIES, PREFIX_ENTRIES = load_liwc_dictionary(
    DICTIONARY_PATH)
PREFIX_LENGTHS = sorted({len(prefix)
                        for prefix in PREFIX_ENTRIES}, reverse=True)

# Symbol-only entries need explicit tokenization; ordinary words preserve apostrophes and hyphens.
SYMBOL_ENTRIES = sorted(
    (token for token in EXACT_ENTRIES if not any(
        character.isalnum() for character in token)),
    key=len,
    reverse=True,
)
symbol_pattern = "|".join(re.escape(token) for token in SYMBOL_ENTRIES)
word_pattern = r"[^\W_]+(?:[-'][^\W_]+)*"
TOKEN_RE = re.compile(f"{symbol_pattern}|{word_pattern}",
                      re.UNICODE | re.IGNORECASE)


def tokenize(text):
    """Yield lowercase LIWC lookup tokens from text."""
    normalized = str(text).lower().replace("’", "'")
    yield from (match.group(0) for match in TOKEN_RE.finditer(normalized))


def category_ids_for_token(token):
    """Return the union of categories from exact and wildcard matches."""
    categories = set(EXACT_ENTRIES.get(token, ()))
    for length in PREFIX_LENGTHS:
        if len(token) >= length:
            categories.update(PREFIX_ENTRIES.get(token[:length], ()))
    return categories


print(f"Loaded {len(CATEGORY_NAMES)} categories, {len(EXACT_ENTRIES)} exact entries, "
      f"and {len(PREFIX_ENTRIES)} wildcard entries.")

Loaded 73 categories, 4071 exact entries, and 2478 wildcard entries.


In [4]:
def analyze_text(text):
    """Analyze one text and return diagnostics plus category rows."""
    tokens = list(tokenize(text))
    word_tokens = [token for token in tokens if any(
        character.isalnum() for character in token)]
    category_counts = Counter()
    matched_word_count = 0
    unmatched_words = Counter()

    for token in tokens:
        category_ids = category_ids_for_token(token)
        is_word = any(character.isalnum() for character in token)
        if category_ids:
            category_counts.update(category_ids)
            if is_word:
                matched_word_count += 1
        elif is_word:
            unmatched_words[token] += 1

    word_count = len(word_tokens)
    rows = []
    for category_id, count in category_counts.most_common():
        rows.append({
            "category_id": category_id,
            "category": CATEGORY_NAMES[category_id],
            "count": count,
            "percent_of_words": 100.0 * count / word_count if word_count else 0.0,
            "percent_of_matched_words": (
                100.0 * count / matched_word_count if matched_word_count else 0.0
            ),
        })

    return {
        "word_count": word_count,
        "matched_word_count": matched_word_count,
        "dictionary_coverage_percent": (
            100.0 * matched_word_count / word_count if word_count else 0.0
        ),
        "unmatched_words": unmatched_words,
        "category_counts": category_counts,
        "rows": rows,
    }


def print_results(result, limit=None):
    """Print a dependency-free category table."""
    print(f"Words: {result['word_count']:,}")
    print(f"Dictionary-matched words: {result['matched_word_count']:,}")
    print(f"Dictionary coverage: {result['dictionary_coverage_percent']:.2f}%")
    print()
    print(f"{'Category':38} {'Count':>8} {'% words':>10} {'% matched':>11}")
    print("-" * 71)
    rows = result["rows"] if limit is None else result["rows"][:limit]
    for row in rows:
        print(f"{row['category'][:38]:38} {row['count']:8d} "
              f"{row['percent_of_words']:10.2f} {row['percent_of_matched_words']:11.2f}")


def analyze_documents(documents):
    """Return one summary record per document for corpus comparisons."""
    records = []
    for document_id, text in enumerate(documents):
        result = analyze_text(text)
        record = {
            "document_id": document_id,
            "word_count": result["word_count"],
            "dictionary_coverage_percent": result["dictionary_coverage_percent"],
        }
        for row in result["rows"]:
            record[row["category"]] = row["percent_of_words"]
        records.append(record)
    return records


def _import_polars():
    try:
        import polars as pl
    except ImportError as error:
        raise ImportError(
            "Polars is optional. Install it with: %pip install polars"
        ) from error
    return pl


def result_to_polars(result):
    """Convert one analyze_text result to a long Polars DataFrame."""
    pl = _import_polars()
    if not result["rows"]:
        return pl.DataFrame(schema={
            "category_id": pl.Int64,
            "category": pl.String,
            "count": pl.Int64,
            "percent_of_words": pl.Float64,
            "percent_of_matched_words": pl.Float64,
        })
    return pl.from_dicts(result["rows"]).sort("percent_of_words", descending=True)


def unmatched_words_to_polars(result):
    """Convert unmatched-word diagnostics to a Polars DataFrame."""
    pl = _import_polars()
    rows = [{"token": token, "count": count}
            for token, count in result["unmatched_words"].most_common()]
    if not rows:
        return pl.DataFrame(schema={"token": pl.String, "count": pl.Int64})
    return pl.from_dicts(rows)


def analyze_documents_polars(documents):
    """Return one row per document with zero-filled LIWC percentage columns."""
    pl = _import_polars()
    category_columns = [CATEGORY_NAMES[key] for key in sorted(CATEGORY_NAMES)]
    records = []
    for document_id, text in enumerate(documents):
        result = analyze_text(text)
        record = {
            "document_id": document_id,
            "word_count": result["word_count"],
            "matched_word_count": result["matched_word_count"],
            "dictionary_coverage_percent": result["dictionary_coverage_percent"],
            **{category: 0.0 for category in category_columns},
        }
        for row in result["rows"]:
            record[row["category"]] = row["percent_of_words"]
        records.append(record)
    if records:
        return pl.from_dicts(records, infer_schema_length=None)
    return pl.DataFrame(schema={
        "document_id": pl.Int64,
        "word_count": pl.Int64,
        "matched_word_count": pl.Int64,
        "dictionary_coverage_percent": pl.Float64,
        **{category: pl.Float64 for category in category_columns},
    })

## Sanity checks

Run these before trusting a result. They verify exact matches, wildcard matches, contractions, case normalization, symbols, and the percentage denominator.


In [5]:
assert "happy" in list(tokenize("HAPPY!"))
assert "can't" in list(tokenize("I can't go"))
assert ":)" in list(tokenize("Great :)"))
assert CATEGORY_NAMES[31] in {CATEGORY_NAMES[value]
                              for value in category_ids_for_token("happy")}
assert CATEGORY_NAMES[35] in {CATEGORY_NAMES[value]
                              for value in category_ids_for_token("abandoned")}

check = analyze_text("happy unknownword")
assert check["word_count"] == 2
assert check["matched_word_count"] == 1
assert check["dictionary_coverage_percent"] == 50.0
assert check["category_counts"][31] == 1
assert next(row for row in check["rows"] if row["category_id"] == 31)[
    "percent_of_words"] == 50.0
print("All sanity checks passed.")

All sanity checks passed.


## Example analysis

The following cells demonstrate a complete analysis of a line-based text file. Each non-empty line in `input.txt` is treated as one independent document, such as one tweet, post, response, or sentence.

Run the examples in order. They will:

1. load and validate the input file;
2. analyze one document in detail;
3. inspect words not covered by the dictionary;
4. analyze every document while preserving document boundaries;
5. optionally create a Polars DataFrame for filtering, aggregation, or export.

Replace `input.txt` with your own path. If one document spans several lines, use a format such as CSV or JSONL instead of this line-based loader.


In [6]:
# STEP 1: Load the corpus.
# Each non-empty line becomes one document and receives a numeric document_id.
# Path.read_text closes the file automatically and makes the encoding explicit.
INPUT_PATH = Path("input.txt")
assert INPUT_PATH.exists(), f"Input file not found: {INPUT_PATH.resolve()}"

documents = [
    line.strip()
    for line in INPUT_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
assert documents, "The input file contains no non-empty documents."

# Expected output: the number of loaded documents and a short preview.
# The preview is truncated so long or sensitive records are not printed in full.
print(f"Loaded {len(documents):,} documents from {INPUT_PATH}.")
print(f"First document preview: {documents[0][:160]!r}")

Loaded 27 documents from input.txt.
First document preview: 'The dev team is bz refactoring #Magento 2 to use jQuery instd of Prototype. 1,300+ changes published 2day on github. http://t.co/AN8bLRuy'


### Step 2: Inspect one document

Start with one document before processing the full corpus. This makes tokenization, coverage, and category assignments easier to verify.

The output begins with the total word count, matched-word count, and dictionary coverage. The table is sorted by category count and shows raw counts plus two percentages. `% words` is the main score to interpret. Because categories overlap, the percentages will not sum to 100%.


In [7]:
# Analyze document 0 only. Change EXAMPLE_DOCUMENT_ID to inspect another row.
EXAMPLE_DOCUMENT_ID = 0
example_text = documents[EXAMPLE_DOCUMENT_ID]
result = analyze_text(example_text)

# Expected output: diagnostics followed by the 25 most frequent LIWC categories.
# Increase or remove limit only when a longer table is useful.
print(f"Document {EXAMPLE_DOCUMENT_ID}: {example_text[:160]!r}\n")
print_results(result, limit=25)

Document 0: 'The dev team is bz refactoring #Magento 2 to use jQuery instd of Prototype. 1,300+ changes published 2day on github. http://t.co/AN8bLRuy'

Words: 25
Dictionary-matched words: 9
Dictionary coverage: 36.00%

Category                                  Count    % words   % matched
-----------------------------------------------------------------------
function (Function Words)                     5      20.00       55.56
prep (Prepositions)                           3      12.00       33.33
work (Work)                                   2       8.00       22.22
focuspresent (Present Focus)                  2       8.00       22.22
verb (Verbs)                                  2       8.00       22.22
cogproc (Cognitive Processes)                 2       8.00       22.22
cause (Causal)                                2       8.00       22.22
relativ (Relativity)                          2       8.00       22.22
article (Articles)                            1       4.00       11.1

### Step 3: Inspect dictionary coverage

Unmatched words explain why coverage may be low. Common misses often include names, URLs, usernames, spelling variants, technical terms, abbreviations, or non-English words. A missing word is not automatically an error; this list is a diagnostic for understanding the data and tokenizer.


In [8]:
# Expected output: up to 20 (token, frequency) pairs for the selected document.
# Frequencies refer only to this document, not the full corpus.
unmatched_preview = result["unmatched_words"].most_common(20)
unmatched_preview

[('dev', 1),
 ('bz', 1),
 ('refactoring', 1),
 ('magento', 1),
 ('2', 1),
 ('jquery', 1),
 ('instd', 1),
 ('prototype', 1),
 ('1', 1),
 ('300', 1),
 ('2day', 1),
 ('github', 1),
 ('http', 1),
 ('t', 1),
 ('co', 1),
 ('an8blruy', 1)]

### Step 4: Analyze all documents

`analyze_documents` returns a list containing one dictionary per document. Each record includes `document_id`, `word_count`, dictionary coverage, and the `% words` value for every category found in that document.

Keeping one record per document is usually preferable for group comparisons and statistical analysis. Categories absent from a plain dictionary record are implicitly zero.


In [9]:
# Run LIWC independently for every non-empty line loaded in Step 1.
results = analyze_documents(documents)

# Expected output: corpus size plus a compact preview of the first record.
# Showing selected fields is easier to read than printing every category at once.
preview_fields = [
    "document_id",
    "word_count",
    "dictionary_coverage_percent",
    "posemo (Positive Emotions)",
    "negemo (Negative Emotions)",
]
first_record_preview = {field: results[0].get(
    field, 0.0) for field in preview_fields}
print(f"Analyzed {len(results):,} documents.")
first_record_preview

Analyzed 27 documents.


{'document_id': 0,
 'word_count': 25,
 'dictionary_coverage_percent': 36.0,
 'posemo (Positive Emotions)': 0.0,
 'negemo (Negative Emotions)': 0.0}

### Step 5: Create a Polars DataFrame

This optional step creates a wide table with one row per document. Metadata columns come first, followed by one float column per LIWC category. Categories not found in a document are explicitly filled with `0.0`.

The full table has many columns, so the example displays only a useful subset. Install Polars in the active notebook kernel with `%pip install polars` if it is unavailable.


In [10]:
try:
    document_frame = analyze_documents_polars(documents)
except ImportError as error:
    # Expected when Polars is not installed; run `%pip install polars`, then rerun.
    print(error)

# Expected shape: number of documents x metadata plus 73 LIWC categories.
print(f"Polars output shape: {document_frame.shape}")
document_frame.head(10)

Polars output shape: (27, 77)


document_id,word_count,matched_word_count,dictionary_coverage_percent,function (Function Words),pronoun (Pronouns),ppron (Personal Pronouns),i (I),we (We),you (You),shehe (SheHe),they (They),ipron (Impersonal Pronouns),article (Articles),prep (Prepositions),auxverb (Auxiliary Verbs),adverb (Adverbs),conj (Conjunctions),negate (Negations),verb (Verbs),adj (Adjectives),compare (Comparisons),interrog (Interrogatives),number (Numbers),quant (Quantifiers),affect (Affect),posemo (Positive Emotions),negemo (Negative Emotions),anx (Anx),anger (Anger),sad (Sad),social (Social),family (Family),friend (Friends),female (Female),male (Male),cogproc (Cognitive Processes),…,tentat (Tentative),certain (Certainty),differ (Differentiation),percept (Perceptual Processes),see (See),hear (Hear),feel (Feel),bio (Biological Processes),body (Body),health (Health),sexual (Sexual),ingest (Ingest),drives (Drives),affiliation (Affiliation),achieve (Achievement),power (Power),reward (Reward),risk (Risk),focuspast (Past Focus),focuspresent (Present Focus),focusfuture (Future Focus),relativ (Relativity),motion (Motion),space (Space),time (Time),work (Work),leisure (Leisure),home (Home),money (Money),relig (Religion),death (Death),informal (Informal Language),swear (Swear),netspeak (Netspeak),assent (Assent),nonflu (Nonfluencies),filler (Filler Words)
i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,25,9,36.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,12.0,4.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,8.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0,4.0,0.0,0.0,0.0,0.0,8.0,0.0,8.0,4.0,4.0,0.0,8.0,4.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0
1,9,5,55.555556,33.333333,11.111111,11.111111,0.0,11.111111,0.0,0.0,0.0,0.0,0.0,0.0,22.222222,0.0,0.0,0.0,33.333333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.333333,11.111111,0.0,0.0,11.111111,0.0,…,0.0,0.0,0.0,11.111111,0.0,11.111111,0.0,0.0,0.0,0.0,0.0,0.0,22.222222,22.222222,0.0,0.0,0.0,0.0,22.222222,11.111111,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,17,13,76.470588,35.294118,5.882353,0.0,0.0,0.0,0.0,0.0,0.0,5.882353,5.882353,11.764706,5.882353,5.882353,0.0,5.882353,11.764706,5.882353,0.0,0.0,0.0,0.0,11.764706,11.764706,0.0,0.0,0.0,0.0,11.764706,0.0,0.0,0.0,0.0,17.647059,…,11.764706,0.0,5.882353,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.529412,0.0,11.764706,5.882353,5.882353,0.0,0.0,17.647059,5.882353,5.882353,0.0,0.0,5.882353,11.764706,5.882353,0.0,5.882353,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2,1,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,50.0,50.0,0.0,0.0
4,16,12,75.0,37.5,18.75,12.5,0.0,6.25,0.0,0.0,6.25,6.25,0.0,6.25,12.5,6.25,0.0,0.0,25.0,6.25,6.25,6.25,0.0,6.25,12.5,6.25,6.25,0.0,0.0,6.25,12.5,0.0,0.0,0.0,0.0,6.25,…,0.0,6.25,0.0,6.25,0.0,6.25,0.0,0.0,0.0,0.0,0.0,0.0,12.5,6.25,6.25,0.0,0.0,0.0,18.75,0.0,0.0,12.5,0.0,6.25,6.25,6.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,28,19,67.857143,32.142857,10.714286,7.142857,3.571429,0.0,3.571429,0.0,0.0,3.571429,3.571429,10.714286,3.571429,0.0,3.571429,0.0,17.857143,0.0,0.0,3.571429,0.0,3.571429,10.714286,7.142857,3.571429,0.0,0.0,3.571429,10.714286,0.0,3.571429,0.0,0.0,3.571429,…,3.571429,0.0,0.0,3.571429,0.0,0.0,3.571429,0.0,0.0,0.0,0.0,0.0,10.714286,3.571429,0.0,3.571429,3.571429,0.0,0.0,10.714286,0.0,3.571429,0.0,3.571429,0.0,7.142857,0.0,0.0,0.0,0.0,0.0,7.142857,0.0,7.142857,3.571429,0.0,0.0
6,16,14,87.5,43.75,18.75,18.75,0.0,6.25,6.25,6.25,0.0,0.0,12.5,12.5,0.0,0.0,0.0,0.0,18.75,0.0,0.0,0.0,0.0,0.0,6.25,6.25,0.